<a href="https://colab.research.google.com/github/AntonyRMV26/INGENIER-A-DEL-CONOCIMIENTO/blob/master/Feature_Extraction_Semana09_MoralesAntony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Semana 09-A — Transfer Learning: Feature Extraction
## Notebook integrado | 🟢 Básico: Malaria + 🟡 Intermedio: Intel Image Classification

**Curso:** IS056B — Ingeniería del Conocimiento  
**Técnica:** **Feature Extraction**: reutilizar un backbone preentrenado en ImageNet, **congelarlo** y entrenar solamente una nueva cabeza clasificadora.

| Nivel | Dataset | Clases | Modelo preentrenado | Secciones del ejemplo de clase |
|---|---|---:|---|---|
| 🟢 Básico | Malaria (TFDS/NIH) | 2: `parasitized`, `uninfected` | **MobileNetV2** | Ejemplo 1: 1.1–1.5 |
| 🟡 Intermedio | Intel Image Classification (Kaggle) | 6 escenas naturales | **EfficientNetB0** | Ejemplo 2: 2.1–2.6 + test |

**No se desarrolla el nivel avanzado** (comparación de backbones y t-SNE), por indicación docente.

> **Precisión de términos:** ambos problemas son de **una sola etiqueta por imagen**; Malaria es *binario* e Intel es *multiclase* (no multietiqueta). No se comparan sus accuracies para decidir qué arquitectura es superior: son problemas diferentes.

## 0. Preparación en Google Colab

1. Abre el archivo `.ipynb` en [Google Colab](https://colab.research.google.com/) y selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4 si está disponible)**.
2. Ejecuta las celdas de arriba hacia abajo. La primera descarga las dependencias que falten; los datasets y pesos de ImageNet se descargan desde Internet.
3. Para verificar el flujo con menor cómputo, activa `MODO_RAPIDO = True`. **Los resultados en este modo no son los resultados finales de la práctica.**
4. Con `MODO_RAPIDO = False`, ejecuta nuevamente desde el principio para obtener los resultados de ambas experiencias.
5. Si deseas conservar checkpoints, gráficas y CSV tras desconectar Colab, configura `USAR_DRIVE = True`. El directorio raíz del experimento contendrá carpetas separadas para Malaria e Intel.

**Nota:** un runtime con GPU puede consumir unidades de cómputo; la cantidad depende del tipo de acelerador y del uso real. Evitamos entrenamientos repetidos innecesarios.

In [ ]:
# Dependencias: instala solo si faltan (compatible con Colab).
import importlib.util
import subprocess
import sys
for paquete, modulo in [('tensorflow-datasets', 'tensorflow_datasets'), ('kagglehub', 'kagglehub')]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
print('✅ Dependencias disponibles')

In [ ]:
import gc
import json
import os
import platform
import random
from pathlib import Path
from collections import Counter

import kagglehub
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

# ================= PANEL ÚNICO DE CONFIGURACIÓN =================
SEED = 42
MODO_RAPIDO = False            # True: ensayo con muestras/épocas reducidas
USAR_DRIVE = False            # True: guardar en Google Drive
GUARDAR_MODELO_COMPLETO = False
BATCH_SIZE = 32
IMG_SIZE_MALARIA = (224, 224)
IMG_SIZE_INTEL = (224, 224)
EPOCHS_MALARIA = 3 if MODO_RAPIDO else 10
EPOCHS_INTEL = 3 if MODO_RAPIDO else 15
AUTOTUNE = tf.data.AUTOTUNE
# ================================================================

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ_SALIDA = Path('/content/drive/MyDrive/FE_Semana09_A_Malaria_Intel')
else:
    RAIZ_SALIDA = Path('/content/FE_Semana09_A_Malaria_Intel') if Path('/content').exists() else Path.cwd() / 'FE_Semana09_A_Malaria_Intel'
OUTPUT_MALARIA = RAIZ_SALIDA / '01_Malaria_Basico'
OUTPUT_INTEL = RAIZ_SALIDA / '02_Intel_Intermedio'
for carpeta in (OUTPUT_MALARIA, OUTPUT_INTEL):
    carpeta.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__, '| TFDS:', tfds.__version__, '| Python:', platform.python_version())
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Modo rápido:', MODO_RAPIDO)
print('Máximo de épocas (Malaria / Intel):', EPOCHS_MALARIA, '/', EPOCHS_INTEL)
print('Carpeta raíz:', RAIZ_SALIDA)

---
# 🟢 NIVEL BÁSICO — MALARIA
## Ejemplo 1 adaptado: clasificación binaria con MobileNetV2

**Pregunta:** ¿cómo aprovechar las características aprendidas en ImageNet para distinguir imágenes de células **parasitadas** y **no parasitadas**?

Mantenemos el flujo del **Ejemplo 1**: cargar imágenes, inspeccionarlas, preprocesar, crear un clasificador con backbone congelado, entrenar, evaluar y visualizar predicciones.

**Limitación metodológica:** las imágenes de TFDS se dividen **por imagen**, no por paciente/lámina. No puede asumirse independencia entre pacientes ni presentarse este modelo como diagnóstico clínico.

### 1.1. Cargar y explorar el dataset

La distribución de TensorFlow Datasets (`malaria`) ofrece dos clases. Como proporciona un único split `train`, reservamos **70 % para entrenamiento**, **15 % para validación** y **15 % para test**. El conjunto de prueba se abre únicamente al final.

In [ ]:
(mal_train_raw, mal_val_raw, mal_test_raw), mal_info = tfds.load(
    'malaria',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    as_supervised=True, with_info=True, shuffle_files=False,
)
MAL_CLASSES = list(mal_info.features['label'].names)
MAL_N_CLASSES = len(MAL_CLASSES)
assert MAL_N_CLASSES == 2 and set(MAL_CLASSES) == {'parasitized', 'uninfected'}
MAL_POS_IDX = MAL_CLASSES.index('parasitized')

if MODO_RAPIDO:
    mal_train_raw = mal_train_raw.shuffle(5000, seed=SEED, reshuffle_each_iteration=False).take(1600)
    mal_val_raw = mal_val_raw.take(400)
    mal_test_raw = mal_test_raw.take(400)

print('Malaria - clases:', list(enumerate(MAL_CLASSES)))
for nombre, ds in [('Train', mal_train_raw), ('Validación', mal_val_raw), ('Test', mal_test_raw)]:
    print(f'  {nombre}: {int(tf.data.experimental.cardinality(ds).numpy()):,} imágenes')

In [ ]:
# EDA: muestras e inspección exploratoria de balance y tamaños originales.
mal_muestras = list(mal_train_raw.take(10).as_numpy_iterator())
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (imagen, label) in zip(axes.flat, mal_muestras):
    ax.imshow(imagen)
    ax.set_title(f'{MAL_CLASSES[int(label)]} | {imagen.shape[1]}×{imagen.shape[0]}')
    ax.axis('off')
plt.suptitle('Malaria — ejemplos del conjunto de entrenamiento', fontsize=15)
plt.tight_layout()
plt.show()

mal_labels_eda, mal_shapes_eda = [], []
for imagen, label in mal_train_raw.take(1000).as_numpy_iterator():
    mal_labels_eda.append(int(label))
    mal_shapes_eda.append((imagen.shape[0], imagen.shape[1]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(MAL_CLASSES, np.bincount(mal_labels_eda, minlength=MAL_N_CLASSES))
axes[0].set(title='Balance (muestra de hasta 1000 imágenes)', ylabel='Cantidad')
mal_altos, mal_anchos = np.array(mal_shapes_eda).T
axes[1].scatter(mal_anchos, mal_altos, alpha=0.30, s=11)
axes[1].set(title='Dimensiones originales (muestra)', xlabel='Ancho', ylabel='Alto')
plt.tight_layout()
plt.show()

### 1.2. Preprocesamiento y Data Augmentation

MobileNetV2 espera píxeles en **[-1, 1]**. Conservamos la proporción de las células usando `resize_with_pad`, entregamos imágenes RGB en `[0, 255]` al modelo y aplicamos el equivalente de `mobilenet_v2.preprocess_input` **dentro de la red**, después del aumento de datos.

**Validación y test no reciben aumentos aleatorios.** Usamos etiquetas *one-hot* y `categorical_crossentropy`, como en el ejemplo de clase.

In [ ]:
def mal_preparar(imagen, etiqueta):
    imagen = tf.image.resize_with_pad(imagen, IMG_SIZE_MALARIA[0], IMG_SIZE_MALARIA[1])
    imagen = tf.cast(imagen, tf.float32)  # [0, 255]; Rescaling ocurrirá dentro del modelo
    etiqueta = tf.one_hot(tf.cast(etiqueta, tf.int32), depth=MAL_N_CLASSES)
    return imagen, etiqueta

mal_train_ds = (mal_train_raw.shuffle(5000, seed=SEED, reshuffle_each_iteration=True)
                .map(mal_preparar, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE).prefetch(AUTOTUNE))
mal_val_ds = (mal_val_raw.map(mal_preparar, num_parallel_calls=AUTOTUNE)
              .batch(BATCH_SIZE).prefetch(AUTOTUNE))
mal_test_ds = (mal_test_raw.map(mal_preparar, num_parallel_calls=AUTOTUNE)
               .batch(BATCH_SIZE).prefetch(AUTOTUNE))

mal_batch_x, mal_batch_y = next(iter(mal_train_ds))
print('Imágenes:', mal_batch_x.shape, '| Etiquetas:', mal_batch_y.shape)
print('Valores antes del preprocesamiento MobileNetV2:',
      float(tf.reduce_min(mal_batch_x)), float(tf.reduce_max(mal_batch_x)))

### 1.3. Construir el modelo (Feature Extraction)

1. Cargamos `MobileNetV2(weights='imagenet', include_top=False)`.
2. **Congelamos** el extractor (`trainable=False`) y usamos `training=False` para BatchNormalization del backbone.
3. Creamos una cabeza propia: **GlobalAveragePooling2D → Dropout → Dense(2, Softmax)**.
4. `RandomFlip/Rotation/Zoom` se activa automáticamente solo en entrenamiento.

In [ ]:
mal_base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(*IMG_SIZE_MALARIA, 3)
)
mal_base.trainable = False
mal_aug = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    layers.RandomRotation(0.08, seed=SEED+1),
    layers.RandomZoom(0.08, seed=SEED+2),
], name='malaria_aumento')

mal_inputs = layers.Input(shape=(*IMG_SIZE_MALARIA, 3), name='celula_rgb_0_255')
x = mal_aug(mal_inputs)
x = layers.Rescaling(1/127.5, offset=-1, name='mobilenetv2_menos1_a_1')(x)
x = mal_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
mal_outputs = layers.Dense(MAL_N_CLASSES, activation='softmax', name='estado_celula')(x)
mal_model = models.Model(mal_inputs, mal_outputs, name='basico_malaria_mobilenetv2')
mal_total_params = mal_model.count_params()
mal_trainable_params = sum(int(np.prod(p.shape)) for p in mal_model.trainable_weights)
assert mal_base.trainable is False
print(f'Parámetros totales: {mal_total_params:,} | Entrenables: {mal_trainable_params:,} | Congelados: {mal_total_params-mal_trainable_params:,}')
mal_model.summary()

### 1.4. Entrenar y visualizar las curvas

Se utiliza **Adam (1e-3)** con **categorical crossentropy**. Además del accuracy, calculamos *precision* y *recall* **específicos de la clase `parasitized`**, importante para detectar errores que podrían pasar inadvertidos con accuracy únicamente. **EarlyStopping** y **ModelCheckpoint** seleccionan por menor `val_loss`.

En modo rápido, los resultados solo verifican que el código funciona; no son conclusiones de rendimiento.

In [ ]:
mal_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(class_id=MAL_POS_IDX, name='precision_parasitized'),
        tf.keras.metrics.Recall(class_id=MAL_POS_IDX, name='recall_parasitized'),
    ],
)
MAL_CHECKPOINT = OUTPUT_MALARIA / 'mejores_pesos_mobilenetv2.weights.h5'
mal_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2,
                                          factor=0.5, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(MAL_CHECKPOINT), monitor='val_loss',
                                        save_best_only=True, save_weights_only=True, verbose=1),
]
mal_history = mal_model.fit(
    mal_train_ds, validation_data=mal_val_ds,
    epochs=EPOCHS_MALARIA, callbacks=mal_callbacks, verbose=1,
)
mal_model.load_weights(str(MAL_CHECKPOINT))
mal_metricas_val = mal_model.evaluate(mal_val_ds, return_dict=True, verbose=0)
print('✅ Validación con los mejores pesos:',
      {k: round(float(v),4) for k,v in mal_metricas_val.items()})
pd.DataFrame(mal_history.history).to_csv(OUTPUT_MALARIA / 'historial_malaria.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
mal_ep = range(1, len(mal_history.history['loss']) + 1)
for ax, metrica, titulo in [(axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')]:
    ax.plot(mal_ep, mal_history.history[metrica], 'o-', label='Entrenamiento')
    ax.plot(mal_ep, mal_history.history['val_'+metrica], 's-', label='Validación')
    ax.set(title=f'Malaria — {titulo}', xlabel='Época', ylabel=titulo)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_MALARIA / 'curvas_malaria.png', dpi=150, bbox_inches='tight')
plt.show()

### 1.5. Evaluación detallada y predicciones visuales

Se muestran **precision, recall y F1 por clase** y una matriz de confusión. Para `parasitized`, los *falsos negativos* son imágenes que realmente están parasitadas, pero el clasificador predice `uninfected`.

Para evitar cualquier desalineación entre imágenes y etiquetas, la inferencia y la lectura de etiquetas ocurren **en el mismo recorrido del dataset**. Esta ampliación de evaluación conserva el espíritu del ejemplo básico.

In [ ]:
def evaluar_malaria(modelo, dataset, nombre):
    reales, predichas, probas = [], [], []
    for imagenes, etiquetas in dataset:
        p = modelo(imagenes, training=False).numpy()
        probas.append(p)
        reales.append(np.argmax(etiquetas.numpy(), axis=1))
        predichas.append(np.argmax(p, axis=1))
    reales = np.concatenate(reales)
    predichas = np.concatenate(predichas)
    probas = np.concatenate(probas)
    cm = confusion_matrix(reales, predichas, labels=range(MAL_N_CLASSES))
    print(f'\n📊 {nombre}: Classification Report')
    print(classification_report(reales, predichas, labels=range(MAL_N_CLASSES),
                                target_names=MAL_CLASSES, digits=4, zero_division=0))
    fn = int(cm[MAL_POS_IDX].sum() - cm[MAL_POS_IDX, MAL_POS_IDX])
    print(f'⚠️ Falsos negativos (parasitized → uninfected): {fn}')
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=MAL_CLASSES, yticklabels=MAL_CLASSES, ax=ax)
    ax.set(title=f'Malaria — {nombre}', xlabel='Predicción', ylabel='Clase real')
    plt.tight_layout()
    plt.savefig(OUTPUT_MALARIA / f'matriz_{nombre.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    report = classification_report(reales, predichas, labels=range(MAL_N_CLASSES),
                                   target_names=MAL_CLASSES, digits=4, output_dict=True,
                                   zero_division=0)
    pd.DataFrame(report).transpose().to_csv(OUTPUT_MALARIA / f'reporte_{nombre.lower()}.csv')
    return reales, predichas, probas, cm

mal_y_val, mal_pred_val, mal_proba_val, mal_cm_val = evaluar_malaria(
    mal_model, mal_val_ds, 'Validacion'
)

### 1.5.1. Test independiente de Malaria

El test se reserva hasta aquí para evaluar **el mismo modelo**, ya seleccionado por validación. Es independiente como partición de imágenes, **no necesariamente por origen clínico**.

In [ ]:
mal_metricas_test = mal_model.evaluate(mal_test_ds, return_dict=True, verbose=0)
mal_y_test, mal_pred_test, mal_proba_test, mal_cm_test = evaluar_malaria(
    mal_model, mal_test_ds, 'Test'
)
print('✅ Métricas del test de Malaria:',
      {k: round(float(v),4) for k,v in mal_metricas_test.items()})

# Mostrar predicciones de test junto con la verdad; sin ocultar la etiqueta real.
mal_ejemplos_x, mal_ejemplos_y = next(iter(mal_test_ds))
mal_ejemplos_p = mal_model(mal_ejemplos_x, training=False).numpy()
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    real = MAL_CLASSES[int(np.argmax(mal_ejemplos_y[i]))]
    pred_idx = int(np.argmax(mal_ejemplos_p[i]))
    pred = MAL_CLASSES[pred_idx]
    ax.imshow(tf.cast(tf.clip_by_value(mal_ejemplos_x[i], 0, 255), tf.uint8).numpy())
    ax.set_title(f'Real: {real}\nPred: {pred} ({mal_ejemplos_p[i,pred_idx]:.1%})',
                 fontsize=10, color='green' if pred==real else 'darkred')
    ax.axis('off')
plt.suptitle('Malaria — predicciones en test', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_MALARIA / 'predicciones_malaria.png', dpi=150, bbox_inches='tight')
plt.show()

### ✅ Resumen del nivel básico

La conclusión debe basarse en **las cifras calculadas**, no en valores esperados. El resultado refleja clasificación de **imágenes celulares**; no constituye diagnóstico médico ni validación clínica.

In [ ]:
RESUMEN_MALARIA = {
    'Dataset': 'Malaria (TFDS)', 'Nivel': 'Básico', 'Modelo': 'MobileNetV2',
    'Clases': MAL_N_CLASSES, 'Epocas': len(mal_history.history['loss']),
    'Parámetros entrenables': mal_trainable_params,
    'Val accuracy': float(mal_metricas_val['accuracy']),
    'Test accuracy': float(mal_metricas_test['accuracy']),
    'Test recall parasitized': float(mal_metricas_test['recall_parasitized']),
    'Test FN parasitized': int(mal_cm_test[MAL_POS_IDX].sum() - mal_cm_test[MAL_POS_IDX, MAL_POS_IDX]),
    'Modo': 'RÁPIDO, NO FINAL' if MODO_RAPIDO else 'COMPLETO',
}
print(pd.Series(RESUMEN_MALARIA).to_string())
pd.DataFrame([RESUMEN_MALARIA]).to_csv(OUTPUT_MALARIA / 'resumen_malaria.csv', index=False)
if GUARDAR_MODELO_COMPLETO:
    mal_model.save(OUTPUT_MALARIA / 'malaria_mobilenetv2_FE.keras')
print('Resultados guardados en:', OUTPUT_MALARIA)

In [ ]:
# Liberar la RAM/GPU del entrenamiento básico antes de cargar EfficientNetB0.
# Las métricas, gráficas y pesos de Malaria ya quedaron guardados en disco.
del mal_model, mal_base, mal_aug
gc.collect()
tf.keras.backend.clear_session()
print('✅ Memoria del modelo básico liberada. Iniciamos el nivel intermedio.')

---
# 🟡 NIVEL INTERMEDIO — INTEL IMAGE CLASSIFICATION
## Ejemplo 2 adaptado: clasificación multiclase con EfficientNetB0

**Problema:** clasificar fotografías en **6 clases excluyentes**: `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`.

**Fuente:** [Intel Image Classification, Kaggle](https://www.kaggle.com/datasets/puneet6060/intel-image-classification). Se utiliza `seg_train` (train + validación), `seg_test` (test independiente) y se excluye `seg_pred` (sin etiquetas).

---
### 2.1. Cargar dataset y EDA

La versión original de Kaggle contiene **seis categorías**: `buildings`, `forest`, `glacier`, `mountain`, `sea` y `street`. Posee aproximadamente **14 034 imágenes etiquetadas para entrenamiento** y **3 000 para prueba**; además hay un directorio de predicción **sin etiquetas**, que se excluye.

Descargamos directamente desde Kaggle con `kagglehub`. Para un dataset público normalmente no necesitas credenciales; si Kaggle solicita autenticación, puedes usar `kagglehub.login()` con tu token de Kaggle (nunca lo escribas en una celda compartida).

In [ ]:
DATASET_ID = 'puneet6060/intel-image-classification'
try:
    dataset_root = Path(kagglehub.dataset_download(DATASET_ID))
except Exception as exc:
    raise RuntimeError(
        'No se pudo descargar Intel desde Kaggle. Comprueba la conexión y las condiciones '
        'de acceso en kaggle.com; si Kaggle solicita iniciar sesión, ejecuta '
        'kagglehub.login() en una celda temporal y vuelve a intentar.'
    ) from exc

EXPECTED_CLASSES = ('buildings', 'forest', 'glacier', 'mountain', 'sea', 'street')

def localizar_directorio_etiquetado(raiz, nombre):
    """Admite tanto seg_train/clases como seg_train/seg_train/clases."""
    candidatos = [raiz / nombre, raiz / nombre / nombre]
    candidatos += sorted((p for p in raiz.rglob(nombre) if p.is_dir()), key=lambda p: len(p.parts))
    for candidato in candidatos:
        if candidato.is_dir():
            subcarpetas = {p.name for p in candidato.iterdir() if p.is_dir()}
            if set(EXPECTED_CLASSES).issubset(subcarpetas):
                return candidato
    raise FileNotFoundError(f'No se encontró el directorio {nombre} con las seis clases en {raiz}')

train_dir = localizar_directorio_etiquetado(dataset_root, 'seg_train')
test_dir = localizar_directorio_etiquetado(dataset_root, 'seg_test')
INTEL_CLASSES = sorted(EXPECTED_CLASSES)
INTEL_N_CLASSES = len(INTEL_CLASSES)

print('✅ Datos descargados en:', dataset_root)
print('✅ Entrenamiento original:', train_dir)
print('✅ Prueba independiente:', test_dir)
print('Clases:', INTEL_CLASSES)

In [ ]:
# EDA 1: número de imágenes y distribución por clase (carpetas etiquetadas)
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

def archivos_imagen(carpeta):
    return sorted(p for p in Path(carpeta).iterdir()
                  if p.is_file() and p.suffix.lower() in EXTENSIONES)

rutas_train = {clase: archivos_imagen(train_dir / clase) for clase in INTEL_CLASSES}
rutas_test  = {clase: archivos_imagen(test_dir / clase) for clase in INTEL_CLASSES}
conteos = pd.DataFrame({
    'Clase': INTEL_CLASSES,
    'Train original': [len(rutas_train[c]) for c in INTEL_CLASSES],
    'Test independiente': [len(rutas_test[c]) for c in INTEL_CLASSES]
})
assert (conteos[['Train original','Test independiente']] > 0).all().all(), 'Hay una clase vacía: revisa los directorios.'
print(conteos.to_string(index=False))
print('\nTotales:', conteos[['Train original', 'Test independiente']].sum().to_dict())

def graficar_conteos(ax, columna, titulo):
    barras = ax.bar(conteos['Clase'], conteos[columna], color=plt.cm.Set3(np.linspace(0, 1, INTEL_N_CLASSES)), edgecolor='#888')
    ax.set_title(titulo, weight='bold')
    ax.set_ylabel('Cantidad de imágenes')
    ax.tick_params(axis='x', rotation=25)
    ax.bar_label(barras, padding=3, fontsize=9)
    ax.margins(y=0.15)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
graficar_conteos(axes[0], 'Train original', 'Distribución — seg_train (antes de dividir)')
graficar_conteos(axes[1], 'Test independiente', 'Distribución — seg_test')
plt.tight_layout()
plt.show()

In [ ]:
# EDA 2: visualizar 4 imágenes por cada clase, como el Ejemplo 2 del docente
fig, axes = plt.subplots(INTEL_N_CLASSES, 4, figsize=(14, 2.55 * INTEL_N_CLASSES))
for i, clase in enumerate(INTEL_CLASSES):
    seleccion = random.Random(SEED + i).sample(rutas_train[clase], k=min(4, len(rutas_train[clase])))
    for j, ruta in enumerate(seleccion):
        with Image.open(ruta) as img:
            axes[i, j].imshow(img.convert('RGB'))
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_title(clase, fontsize=12, weight='bold', loc='left')
plt.suptitle('Intel — Muestras de las seis clases', y=1.0, fontsize=15, weight='bold')
plt.tight_layout()
plt.show()

### 2.1.1. Crear particiones de entrenamiento, validación y prueba

El **80 %** de `seg_train` se destina a entrenamiento y el **20 %** a validación usando la misma semilla; `seg_test` se carga aparte. La separación es **por archivo, no estratificada**; verificamos arriba la distribución general. **Nunca** se usa `seg_test` para seleccionar épocas o modificar el modelo.

Usamos `label_mode='categorical'`, que genera vectores *one-hot* para las seis clases, como corresponde a `categorical_crossentropy` + `softmax`.

In [ ]:
# Cargador de Keras: imágenes redimensionadas a 224 × 224 y etiquetas one-hot
intel_train_raw, intel_val_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.20,
    subset='both',
    seed=SEED,
    shuffle=True,
    image_size=IMG_SIZE_INTEL,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=INTEL_CLASSES,
)
intel_test_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    image_size=IMG_SIZE_INTEL,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=INTEL_CLASSES,
)

print('Clases confirmadas:', intel_train_raw.class_names)
print('Batches train / val / test:',
      int(tf.data.experimental.cardinality(intel_train_raw)),
      int(tf.data.experimental.cardinality(intel_val_raw)),
      int(tf.data.experimental.cardinality(intel_test_raw)))

# Acceso ligero a metadatos: no decodifica todas las imágenes para contarlas
if hasattr(intel_train_raw, 'file_paths') and hasattr(intel_val_raw, 'file_paths'):
    for nombre, ds in [('Train', intel_train_raw), ('Validación', intel_val_raw)]:
        distribucion = Counter(Path(p).parent.name for p in ds.file_paths)
        print(f'{nombre}: {len(ds.file_paths):,} imágenes |',
              {clase: distribucion[clase] for clase in INTEL_CLASSES})

if MODO_RAPIDO:
    print('⚠️ MODO_RAPIDO: entrenaremos con un subconjunto de batches; no usar estas métricas como resultado final.')
    intel_train_raw = intel_train_raw.take(60)
    intel_val_raw = intel_val_raw.take(16)

---
### 2.2. Preprocesamiento con Data Augmentation

Como en el Ejemplo 2, aplicamos `RandomFlip`, `RandomRotation(0.15)`, `RandomZoom(0.15)` y `RandomContrast(0.1)` **solo a entrenamiento**. Validación y test mantienen las imágenes originales (salvo redimensionamiento).

**Detalle crítico:** EfficientNetB0 de `tf.keras.applications` ya realiza su propia reescalación interna. Debe recibir imágenes `float32` en **[0, 255]**. Por tanto, **NO dividimos los píxeles por 255** antes del backbone; `efficientnet.preprocess_input` es una operación identidad en esta arquitectura.

In [ ]:
# Augmentation idéntico al utilizado en el ejemplo intermedio de clase
intel_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='augmentation_intel')

# Entrenamiento: augmentations nuevas durante cada recorrido del dataset.
# No usamos .cache() después del aumento: eso congelaría las transformaciones.
def preprocess_train(images, labels):
    images = intel_augmentation(tf.cast(images, tf.float32), training=True)
    images = tf.keras.applications.efficientnet.preprocess_input(images)  # identidad para EfficientNetB0
    return images, labels

def preprocess_eval(images, labels):
    return tf.keras.applications.efficientnet.preprocess_input(tf.cast(images, tf.float32)), labels

intel_train_ds = intel_train_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
intel_val_ds = intel_val_raw.map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
intel_test_ds = intel_test_raw.map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

x_batch, y_batch = next(iter(intel_train_ds))
print('Forma de imágenes:', x_batch.shape, '| One-hot:', y_batch.shape)
print('Rango de píxeles en el batch:',
      f'{float(tf.reduce_min(x_batch)):.2f} a {float(tf.reduce_max(x_batch)):.2f}')

In [ ]:
# Mostrar la fotografía original frente a 9 versiones transformadas
sample_imgs, sample_labels = next(iter(intel_train_raw))
original = sample_imgs[0]
clase_original = INTEL_CLASSES[int(tf.argmax(sample_labels[0]))]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes.flat[0].imshow(tf.cast(tf.clip_by_value(original, 0, 255), tf.uint8).numpy())
axes.flat[0].set_title('Original', weight='bold')
axes.flat[0].axis('off')
for i, ax in enumerate(axes.flat[1:], start=1):
    alterada = intel_augmentation(tf.expand_dims(original, axis=0), training=True)[0]
    ax.imshow(tf.cast(tf.clip_by_value(alterada, 0, 255), tf.uint8).numpy())
    ax.set_title(f'Augmentation #{i}')
    ax.axis('off')
plt.suptitle(f'Data Augmentation — {clase_original}', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

---
### 2.3. Construir modelo con EfficientNetB0

**Feature Extraction**: `weights='imagenet'` aprovecha características aprendidas en ImageNet; `include_top=False` quita su clasificador original, y `base_model.trainable=False` congela el backbone.

Reproducimos la cabeza del docente: **GlobalAveragePooling2D → BatchNormalization → Dropout(0.3) → Dense(128, ReLU) → Dropout(0.2) → Dense(6, Softmax)**.

In [ ]:
intel_base = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE_INTEL, 3)
)
intel_base.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE_INTEL, 3), name='imagen_intel')
x = intel_base(inputs, training=False)  # BatchNorm del backbone en inferencia
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(INTEL_N_CLASSES, activation='softmax', name='escena')(x)

intel_model = models.Model(inputs, outputs, name='intel_efficientnetb0_feature_extraction')
total = intel_model.count_params()
trainable = sum(int(np.prod(w.shape)) for w in intel_model.trainable_weights)
print(f'📐 Total: {total:,} | 🔓 Entrenables: {trainable:,} | 🔒 Congelados: {total - trainable:,}')
assert intel_base.trainable is False, 'La base debe mantenerse congelada en Feature Extraction.'
intel_model.summary()

---
### 2.4. Entrenar con callbacks profesionales

Igual que el Ejemplo 2: **Adam (1e-3), pérdida categorical crossentropy, accuracy, EarlyStopping y ReduceLROnPlateau**.

La reducción de LR ocurre si no mejora `val_loss` durante 2 épocas; `EarlyStopping` detiene antes de sobreajustar y restaura los pesos con mejor `val_loss`. También guardamos un *checkpoint* en disco para protegernos de cortes de sesión.

In [ ]:
intel_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

INTEL_CHECKPOINT = OUTPUT_INTEL / 'intel_effnetb0_mejores_pesos.weights.h5'
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(INTEL_CHECKPOINT), monitor='val_loss', save_best_only=True,
        save_weights_only=True, verbose=1)
]

intel_history = intel_model.fit(
    intel_train_ds,
    validation_data=intel_val_ds,
    epochs=EPOCHS_INTEL,
    callbacks=callbacks,
    verbose=1,
)

# ModelCheckpoint guarda los mejores pesos por val_loss, el mismo criterio del EarlyStopping.
# Esto también permite restituirlos si una nueva sesión de Colab se interrumpe.
print('\n🏆 Mayor val_accuracy histórica:', f"{max(intel_history.history['val_accuracy']):.4f}")
intel_metricas_val = intel_model.evaluate(intel_val_ds, return_dict=True, verbose=0)
print('✅ Métricas actuales del modelo restaurado:',
      {k: round(float(v), 4) for k, v in intel_metricas_val.items()})
pd.DataFrame(intel_history.history).to_csv(OUTPUT_INTEL / 'historial_entrenamiento.csv', index=False)

In [ ]:
# Curvas de aprendizaje: accuracy y loss para train / validación
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, len(intel_history.history['accuracy']) + 1)
axes[0].plot(epochs_x, intel_history.history['accuracy'], 'o-', label='Entrenamiento')
axes[0].plot(epochs_x, intel_history.history['val_accuracy'], 's-', label='Validación')
axes[0].set(title='Accuracy — Intel (EfficientNetB0 FE)', xlabel='Época', ylabel='Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, intel_history.history['loss'], 'o-', label='Entrenamiento')
axes[1].plot(epochs_x, intel_history.history['val_loss'], 's-', label='Validación')
axes[1].set(title='Loss — Intel (EfficientNetB0 FE)', xlabel='Época', ylabel='Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_INTEL / 'curvas_aprendizaje.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 2.5. Evaluación detallada: Classification Report + Confusion Matrix

Repetimos la evaluación del Ejemplo 2 **en validación**. El reporte muestra **precision, recall, f1-score y support** de cada categoría. La matriz utiliza **filas = clase real, columnas = predicción**.

**Cuidado técnico:** para evitar desalineación si el dataset cambia el orden de los ejemplos, calculamos predicciones y etiquetas **dentro del mismo recorrido de batches**.

In [ ]:
def predecir_con_etiquetas(modelo, dataset):
    verdaderas, predichas, probabilidades = [], [], []
    for images, labels in dataset:
        probs = modelo(images, training=False).numpy()
        verdaderas.append(np.argmax(labels.numpy(), axis=1))
        predichas.append(np.argmax(probs, axis=1))
        probabilidades.append(probs)
    return (np.concatenate(verdaderas),
            np.concatenate(predichas),
            np.concatenate(probabilidades))

y_val_true, y_val_pred, prob_val = predecir_con_etiquetas(intel_model, intel_val_ds)
print('📊 Classification Report — Intel, VALIDACIÓN')
print(classification_report(y_val_true, y_val_pred,
                            labels=range(INTEL_N_CLASSES), target_names=INTEL_CLASSES,
                            digits=4, zero_division=0))

report_val = classification_report(y_val_true, y_val_pred, labels=range(INTEL_N_CLASSES),
                                    target_names=INTEL_CLASSES, digits=4,
                                    output_dict=True, zero_division=0)
pd.DataFrame(report_val).transpose().to_csv(OUTPUT_INTEL / 'classification_report_validacion.csv')

In [ ]:
def dibujar_confusion(verdaderas, predichas, titulo, archivo):
    cm = confusion_matrix(verdaderas, predichas, labels=range(INTEL_N_CLASSES))
    fig, ax = plt.subplots(figsize=(9, 7.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=INTEL_CLASSES, yticklabels=INTEL_CLASSES,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set(xlabel='Clase predicha', ylabel='Clase real', title=titulo)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_INTEL / archivo, dpi=150, bbox_inches='tight')
    plt.show()
    return cm

cm_val = dibujar_confusion(y_val_true, y_val_pred,
                           'Matriz de confusión — VALIDACIÓN', 'matriz_validacion.png')

confusiones = sorted(((int(cm_val[i, j]), INTEL_CLASSES[i], INTEL_CLASSES[j])
                      for i in range(INTEL_N_CLASSES) for j in range(INTEL_N_CLASSES)
                      if i != j and cm_val[i, j] > 0), reverse=True)
print('🔎 Confusiones más frecuentes (real → predicha):')
for cantidad, real, pred in confusiones[:8]:
    print(f'  {real} → {pred}: {cantidad} imágenes')

---
### 2.6. Visualización de Feature Maps

Siguiendo la práctica, examinamos las activaciones de cuatro capas de **EfficientNetB0** para una imagen de validación. Mostramos la imagen original y los primeros ocho canales de cada salida.

Las primeras capas tienden a describir estructuras sencillas; las últimas contienen representaciones más abstractas. **Estos mapas son activaciones internas, no una explicación causal ni mapas de atención del modelo.**

In [ ]:
layer_names = [
    'block2a_expand_conv',
    'block4a_expand_conv',
    'block6a_expand_conv',
    'top_conv'
]

# Comprobar que cada capa esté disponible en la versión de Keras del entorno
capas_disponibles = {layer.name for layer in intel_base.layers}
assert all(n in capas_disponibles for n in layer_names), (
    'Los nombres de capas de EfficientNet cambiaron: revisa intel_base.summary().')

# Extraer una imagen sin data augmentation
muestra_imgs, muestra_etiquetas = next(iter(intel_val_raw))
muestra_original = muestra_imgs[0]
muestra_clase = INTEL_CLASSES[int(tf.argmax(muestra_etiquetas[0]))]
muestra_batch = tf.expand_dims(tf.cast(muestra_original, tf.float32), axis=0)

fig, axes = plt.subplots(len(layer_names), 9,
                         figsize=(18, 2.6 * len(layer_names)))
for fila, nombre_capa in enumerate(layer_names):
    extractor = models.Model(inputs=intel_base.input,
                             outputs=intel_base.get_layer(nombre_capa).output)
    mapas = extractor(muestra_batch, training=False).numpy()
    axes[fila, 0].imshow(tf.cast(tf.clip_by_value(muestra_original, 0, 255), tf.uint8).numpy())
    axes[fila, 0].set_ylabel(nombre_capa.replace('_', '\n'), rotation=0,
                             labelpad=65, weight='bold', va='center')
    if fila == 0:
        axes[fila, 0].set_title('Original')
    axes[fila, 0].axis('off')
    for col in range(1, 9):
        ax = axes[fila, col]
        ax.imshow(mapas[0, :, :, col - 1], cmap='viridis')
        ax.axis('off')
        if fila == 0:
            ax.set_title(f'Canal {col}')
plt.suptitle(f'Feature Maps — EfficientNetB0 | Imagen: {muestra_clase}',
             weight='bold', fontsize=15)
plt.tight_layout()
plt.savefig(OUTPUT_INTEL / 'feature_maps.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 2.7. Extensión metodológica: prueba final independiente

A diferencia del notebook de flores, **Intel ya proporciona `seg_test` con etiquetas**. Lo usamos aquí una sola vez para estimar el rendimiento final de nuestro modelo (después de entrenar y validar), sin reajustar hiperparámetros con sus resultados.

En modo rápido, **estos resultados son únicamente de comprobación**, no un experimento concluido.

In [ ]:
intel_metricas_test = intel_model.evaluate(intel_test_ds, return_dict=True, verbose=1)
y_test_true, y_test_pred, prob_test = predecir_con_etiquetas(intel_model, intel_test_ds)
print('\n🧪 Classification Report — TEST INDEPENDIENTE')
print(classification_report(y_test_true, y_test_pred,
                            labels=range(INTEL_N_CLASSES), target_names=INTEL_CLASSES,
                            digits=4, zero_division=0))
cm_test = dibujar_confusion(y_test_true, y_test_pred,
                            'Matriz de confusión — TEST INDEPENDIENTE',
                            'matriz_test.png')

report_test = classification_report(y_test_true, y_test_pred,
                                     labels=range(INTEL_N_CLASSES), target_names=INTEL_CLASSES,
                                     digits=4, output_dict=True, zero_division=0)
pd.DataFrame(report_test).transpose().to_csv(OUTPUT_INTEL / 'classification_report_test.csv')
pd.DataFrame(cm_test, index=INTEL_CLASSES, columns=INTEL_CLASSES).to_csv(
    OUTPUT_INTEL / 'matriz_test.csv')
print('✅ Test final:', {k: round(float(v), 4) for k, v in intel_metricas_test.items()})
if MODO_RAPIDO:
    print('⚠️ Experimento de prueba rápida: cambia MODO_RAPIDO=False y ejecuta todo para la versión final.')

In [ ]:
# Predicciones visuales (útiles para explicar errores en la exposición)
imagenes_ejemplo, etiquetas_ejemplo = next(iter(intel_test_ds))
probas_ejemplo = intel_model(imagenes_ejemplo, training=False).numpy()
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(tf.cast(tf.clip_by_value(imagenes_ejemplo[i], 0, 255), tf.uint8).numpy())
    real = INTEL_CLASSES[int(tf.argmax(etiquetas_ejemplo[i]))]
    indice_pred = int(np.argmax(probas_ejemplo[i]))
    prediccion = INTEL_CLASSES[indice_pred]
    confianza = float(probas_ejemplo[i, indice_pred])
    ax.set_title(f'Real: {real}\nPred: {prediccion} ({confianza:.1%})',
                 fontsize=10, color='green' if real == prediccion else 'darkred')
    ax.axis('off')
plt.suptitle('Ejemplos de predicción — Test independiente', weight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_INTEL / 'predicciones_visuales.png', dpi=150, bbox_inches='tight')
plt.show()

---
### ✅ Conclusión del Ejemplo 2

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════╗
  INTERMEDIO — INTEL IMAGE CLASSIFICATION (SEMANA 09-A)
╚══════════════════════════════════════════════════════╝
Dataset: Kaggle Intel | Clases: {INTEL_N_CLASSES} ({', '.join(INTEL_CLASSES)})
Arquitectura: EfficientNetB0 (ImageNet) + clasificador nuevo
Backbone congelado: {not intel_base.trainable}
Parámetros entrenables: {trainable:,} de {total:,}
Data Augmentation: flip, rotation, zoom y contrast (solo train)
Máximo de épocas: {EPOCHS_INTEL} | Épocas efectivas: {len(intel_history.history['loss'])}
Mayor accuracy histórica en validación: {max(intel_history.history['val_accuracy']):.4f}
Accuracy del modelo restaurado (val): {intel_metricas_val['accuracy']:.4f}
Accuracy final (test independiente): {intel_metricas_test['accuracy']:.4f}
Loss final (test independiente): {intel_metricas_test['loss']:.4f}
Tipo de ejecución: {'PRUEBA RÁPIDA — NO FINAL' if MODO_RAPIDO else 'COMPLETA'}
Carpeta con resultados: {OUTPUT_INTEL}
""")

# Guardar pesos y, opcionalmente, el modelo completo.
print('Pesos del mejor checkpoint:', INTEL_CHECKPOINT)
if GUARDAR_MODELO_COMPLETO:
    archivo_modelo = OUTPUT_INTEL / 'intel_efficientnetb0_FE.keras'
    intel_model.save(archivo_modelo)
    print('Modelo completo guardado:', archivo_modelo)
print('✅ Resultados tabulares e imágenes exportados.')

In [ ]:
# Resumen final de ambas experiencias, una vez ejecutadas en orden.
RESUMEN_INTEL = {
    'Dataset':'Intel (Kaggle)', 'Nivel':'Intermedio', 'Modelo':'EfficientNetB0',
    'Clases':INTEL_N_CLASSES, 'Epocas':len(intel_history.history['loss']),
    'Parámetros entrenables':trainable,
    'Val accuracy':float(intel_metricas_val['accuracy']),
    'Test accuracy':float(intel_metricas_test['accuracy']),
    'Modo':'RÁPIDO, NO FINAL' if MODO_RAPIDO else 'COMPLETO',
}
comparacion_descriptiva = pd.DataFrame([RESUMEN_MALARIA, RESUMEN_INTEL])
display(comparacion_descriptiva)
comparacion_descriptiva.to_csv(RAIZ_SALIDA / 'resumen_ambas_experiencias.csv', index=False)
print('✅ Notebook concluido: básico e intermedio. No se ejecutó nivel avanzado.')
print('⚠️ Las accuracies pertenecen a datasets distintos; NO son un benchmark entre backbones.')
print('Resultados:', RAIZ_SALIDA)